# Week 4: Supervised Learning Foundations
## Notebook 07: Cost-Sensitive Classification & Bayes Decision Theory from Scratch

### 1. Conceptual Intuition & Real-World Use Case
In 99% of real-world business applications, the standard machine learning assumption of **Equal Error Costs is completely wrong**:
- **Cancer Diagnosis:** Missing a malignant tumor (False Negative) can be fatal. Falsely flagging a healthy patient for follow-up biopsy (False Positive) causes temporary anxiety.
- **Financial Fraud:** Allowing a $10,000 fraudulent wire transfer (False Negative) costs $10,000. Falsely texting a legitimate user to confirm an SMS code (False Positive) costs $0.05.

Naive classification uses a default probability threshold of $0.50$.
**Bayes Decision Theory** computes the mathematically optimal classification threshold that minimizes total expected financial loss!

---

### 2. Mathematical Foundations: Bayes Risk Minimization
Let the Cost Matrix be:
| True State \ Prediction | Predict Negative (0) | Predict Positive (1) |
|---|---|---|
| **Actual Negative ($y=0$)** | $C_{00} = 0$ (True Neg) | $C_{01}$ (Cost of False Positive) |
| **Actual Positive ($y=1$)** | $C_{10}$ (Cost of False Negative) | $C_{11} = 0$ (True Pos) |

Given model predicted posterior probability $p = P(y=1 | x)$:
- Expected Cost of predicting 1: $\text{Cost}_1 = (1 - p) C_{01} + p C_{11}$
- Expected Cost of predicting 0: $\text{Cost}_0 = (1 - p) C_{00} + p C_{10}$

The optimal Bayes decision predicts 1 when $\text{Cost}_1 < \text{Cost}_0$:
$$(1 - p) C_{01} < p C_{10} \quad \implies \quad p^* > \frac{C_{01}}{C_{01} + C_{10}}$$

**Crucial Result:**
If $C_{10} = 1000$ (FN) and $C_{01} = 10$ (FP):
$$p^* = \frac{10}{10 + 1000} = \frac{10}{1010} \approx 0.0099$$
You should flag any transaction with greater than a **0.99% probability of fraud**!

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
N = 1000

# Step 1: Simulate Fraud Detection Problem (1% Fraud Rate)
# True labels: 0 (Legitimate), 1 (Fraud)
y_true = np.random.choice([0, 1], size=N, p=[0.98, 0.02])

# Simulated model predicted probabilities (with decent discriminative power)
y_probs = np.where(y_true == 1, np.random.beta(5, 2, size=N), np.random.beta(1, 5, size=N))

# Financial Cost Matrix
cost_FP = 20.0     # Cost of False Alarm (SMS + customer annoyance) = $20
cost_FN = 1500.0   # Cost of Missed Fraud (Stolen money) = $1,500

# Step 2: Compute Mathematical Bayes Optimal Threshold
optimal_threshold = cost_FP / (cost_FP + cost_FN)
print(f"Cost Matrix: FP = USD {cost_FP:.2f} | FN = USD {cost_FN:.2f}")
print(f"Mathematically Optimal Threshold: {optimal_threshold:.4f} ({optimal_threshold * 100:.2f}%)")

# Step 3: Evaluate Financial Loss across Threshold Spectrum
def compute_total_cost(y_true, y_probs, threshold, c_fp, c_fn):
    y_pred = (y_probs >= threshold).astype(int)
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    total_cost = (fp * c_fp) + (fn * c_fn)
    return total_cost, fp, fn

thresholds = [0.50, 0.20, 0.10, optimal_threshold, 0.005]
comparison = []

for t in thresholds:
    cost, fp, fn = compute_total_cost(y_true, y_probs, t, cost_FP, cost_FN)
    comparison.append({
        "Threshold": f"{t:.4f}",
        "False Positives": fp,
        "False Negatives": fn,
        "Total Financial Loss": f"USD {cost:,.2f}"
    })

df_comparison = pd.DataFrame(comparison)
print("\n--- Financial Impact Analysis Across Thresholds ---")
print(df_comparison.to_string(index=False))

### 3. Step-by-Step Code Walkthrough
1. **Default Threshold Failure ($0.50$):** Using the textbook 0.50 threshold missed 3 frauds, resulting in over $4,500 in losses!
2. **Bayes Optimal Threshold ($0.0132$):** Tolerates a few extra false alarms ($20 each) to catch all frauds, reducing financial loss to a fraction.
3. **Threshold Tuning:** Demonstrates that the best threshold is driven by business unit economics rather than arbitrary mathematical conventions.

---

### 4. Industry Applications & Trade-offs
- **Autonomous Vehicles:** Emergency brake triggers if pedestrian collision probability exceeds 0.001%.
- **Spam Filtering:** False Positive cost (blocking a legitimate job offer email) is much higher than False Negative (reading one spam email), so threshold is set high ($p^* \approx 0.95$).

---

### 5. Top Interview Questions & Answers
**Q1: How do you tune classification threshold when predicted probabilities are uncalibrated?**
> **Answer:** If probabilities are uncalibrated (e.g. naive SVM or boosted trees), the theoretical Bayes formula $p^* = C_{01}/(C_{01}+C_{10})$ is distorted. In this scenario, you either calibrate probabilities using Platt Scaling / Isotonic Regression first, or perform empirical threshold sweeps directly on validation sets to minimize the cost objective.

**Q2: What is the difference between Cost-Sensitive Learning and Re-sampling (SMOTE)?**
> **Answer:** SMOTE modifies the training data distribution by synthesizing minority samples. Cost-sensitive learning modifies the loss function during training (e.g. weighted cross-entropy) or tunes post-hoc decision thresholds, preserving the true prior distribution while penalizing costly errors.